# Setup and API Key and Configuration
Install the elsapy library and configure the API key required for authentication with Elsevier's APIs.

In [1]:
# Import necessary libraries
import json
import csv

from elsapy.elsclient import ElsClient
from elsapy.elssearch import ElsSearch

In [2]:
# Load API key from a configuration file
with open("config.json") as config_file:
    config = json.load(config_file)

# Initialize the ElsClient with the API key
client = ElsClient(config['apikey'])

# Initialize the Elsevier Client
Create an Elsevier client instance using your API key and configure the connection settings.

In [3]:
# Set the connection settings for the client
client.base_url = 'https://api.elsevier.com/content/search/scopus'

# Verify the client connection
if client:
    print("Client initialized successfully.")
else:
    print("Failed to initialize client.")

Client initialized successfully.


# Search for Articles by Keyword
Demonstrate how to search for articles in Scopus using keywords and how to handle the response data.

In [ ]:
# Import necessary libraries for searching articles

# Define the search query using a keyword
# search_query = (
#     'TITLE-ABS-KEY('
#     '("smart cit*" OR "intelligent cit*" OR "digital city" OR '
#     '"smart urban" OR "urban computing") '
#     "AND "
#     '("generative AI" OR "generative artificial intelligence" OR '
#     '"GAN" OR "generative adversarial network*" OR "diffusion model*" OR '
#     '"large language model*" OR "LLM" OR "LLMs" OR '
#     '"variational autoencoder*" OR "VAE*" OR "autoregressive model*" OR '
#     '"transformer*" OR "retrieval-augmented generation" OR "RAG" OR "AI agent*")'
#     ") "
#     "AND PUBYEAR > 2018 AND PUBYEAR < 2026 "
#     'AND LIMIT-TO(LANGUAGE, "English")'
# )

search_query = (
    'TITLE-ABS-KEY( '
      '( sensor* OR sensing OR "internet of things" '
        'OR "street view" OR "street-level imag*" OR "streetscape imag*" '
        'OR "computer vision" OR "deep learning" '
        'OR "semantic segmentation" OR "image segmentation" '
        'OR "image classification" OR "object detection" '
        'OR "LiDAR" OR "point cloud*" OR photogrammetr* '
        'OR "GPS" OR "GNSS" '
        'OR wearable* OR acceleromet* OR camera* OR smartphone* ) '
      'AND '
      '( walkabilit* OR walkable '
        'OR "pedestrian environment" OR "pedestrian infrastructure" '
        'OR sidewalk OR footpath OR streetscape ) '
    ') '
    'AND PUBYEAR > 2015 AND PUBYEAR < 2027 '
    'AND DOCTYPE(ar) AND SRCTYPE(j) AND LANGUAGE(english)'
)

print(search_query)

# Initialize the ElsSearch object with the search query and client
doc_srch = ElsSearch(search_query, 'scopus')

# Execute the search
doc_srch.execute(client, get_all=True)

# Check if the search was successful
if doc_srch.results:
    print(f"Found {len(doc_srch.results)} results.")
else:
    print("No results found.")

TITLE-ABS-KEY( ( sensor* OR sensing OR "internet of things" OR "street view" OR "street-level imag*" OR "streetscape imag*" OR "computer vision" OR "deep learning" OR "semantic segmentation" OR "image segmentation" OR "image classification" OR "object detection" OR "LiDAR" OR "point cloud*" OR photogrammetr* OR "GPS" OR "GNSS" OR wearable* OR acceleromet* OR camera* OR smartphone* ) AND ( walkabilit* OR walkable OR "pedestrian environment" OR "pedestrian infrastructure" OR sidewalk OR footpath OR streetscape ) ) AND PUBYEAR > 2015 AND PUBYEAR < 2027 AND DOCTYPE(ar) AND SRCTYPE(j) AND LANGUAGE(english)
Found 1466 results.


# Save the results to a CSV file

In [5]:
# Function to export results to CSV with dynamic fields
def export_to_csv(data, filename):
    # Get all possible keys across all dictionaries
    all_keys = set()
    for item in data:
        all_keys.update(item.keys())

    # Convert to sorted list for consistent column order
    fieldnames = sorted(list(all_keys))

    with open(filename, "w", newline="", encoding="utf-8") as output_file:
        dict_writer = csv.DictWriter(output_file, fieldnames=fieldnames)
        dict_writer.writeheader()
        dict_writer.writerows(data)

    print(f"CSV exported {len(data)} records")


# Function to export results to JSON
def export_to_json(data, filename):
    with open(filename, "w", encoding="utf-8") as output_file:
        json.dump(data, output_file, indent=4, ensure_ascii=False)

    print(f"JSON exported with {len(data)} records")

In [6]:
# Filter out documents with missing doi
results = [doc for doc in doc_srch.results if doc.get("prism:doi")]

# Export the results to a CSV file
# export_to_csv(results, "../data/01_scopus_results.csv")
export_to_json(results, "../data/01_scopus_results.json")

JSON exported with 1448 records


In [15]:
import json
import re
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

p1 = project_root / "data" / "01_scopus_results.json"
p2 = project_root / "data" / "02_document_search_results.json"
for p in [p1, p2]:
    if not p.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {p}")


def normalize_scopus_id(value):
    if value is None:
        return None
    if isinstance(value, list):
        if not value:
            return None
        return normalize_scopus_id(value[0])

    s = str(value).strip()
    if not s:
        return None

    s = s.replace("2-s2.0-", "").replace("SCOPUS_ID:", "")
    if s.lower().startswith("scopus_id:"):
        s = s.split(":", 1)[1]

    m = re.search(r"(\d+)", s)
    return m.group(1) if m else None


def extract_scopus_id(record):
    if not isinstance(record, dict):
        return None

    for key in ["scopus_id", "eid", "dc:identifier", "SCOPUS_ID"]:
        v = record.get(key)
        sid = normalize_scopus_id(v)
        if sid:
            return sid

    for key in ["prism:url", "@href"]:
        v = record.get(key)
        if isinstance(v, str):
            m = re.search(r"scopus_id/(\d+)|scp=(\d+)", v)
            if m:
                return m.group(1) or m.group(2)

    for item in record.get("link", []):
        if isinstance(item, dict):
            href = item.get("@href")
            if isinstance(href, str):
                m = re.search(r"scopus_id/(\d+)|scp=(\d+)", href)
                if m:
                    return m.group(1) or m.group(2)

    return None


with open(p1, "r", encoding="utf-8") as f:
    d1 = json.load(f)

with open(p2, "r", encoding="utf-8") as f:
    d2 = json.load(f)

ids_01 = {extract_scopus_id(x) for x in d1 if extract_scopus_id(x)}
ids_02 = {extract_scopus_id(x) for x in d2 if extract_scopus_id(x)}
missing_in_02 = sorted(ids_01 - ids_02)

print(f"01_scopus_results.json: {len(ids_01)} IDs")
print(f"02_document_search_results.json: {len(ids_02)} IDs")
print(f"Faltando em 02: {len(missing_in_02)}")

if missing_in_02:
    print("IDs ausentes em 02:")
    print(missing_in_02)
else:
    print("Todos os IDs de 01 estão em 02.")

    allowed_ids = ids_01
    filtered_02 = []
    for item in d2:
        sid = extract_scopus_id(item)
        if sid in allowed_ids:
            filtered_02.append(item)

    with open(p2, "w", encoding="utf-8") as f:
        json.dump(filtered_02, f, ensure_ascii=False, indent=4)

    print(f"Arquivo 02 reescrito com {len(filtered_02)} registros.")


01_scopus_results.json: 1447 IDs
02_document_search_results.json: 1591 IDs
Faltando em 02: 6
IDs ausentes em 02:
['105046083709', '105046196836', '105046256132', '105046296852', '105046342171', '105046474851']
